# SafeStack — Phase 3 FU5c: SFT train → dev-select on Colab (A100)

Trains the LoRA/QLoRA SFT adapter (ADR-0015 dec.3), uploads it to a **private** HF-Hub repo for an
immutable id (dec.7b), then runs **rule-9 checkpoint selection** on the held-out DEV suites
(dec.4 + amendment 1): the bare SFT policy (no guardrails = the **C5 condition**) vs the frozen base.

Selection rule (amendment 1): reject a checkpoint whose dev-helpfulness **answer-rate** drops more
than 0.10 below base (mode-collapse tripwire); among survivors minimise `ASR_dev + over_refusal_dev`.
This 1-epoch recipe has a single checkpoint, so selection is chiefly the tripwire gate.

**Committed (aggregate-only):** the `Selection` artifact, the dev metrics, the loss curves, and this
executed notebook. **Private, never committed:** the adapter weights (a private HF-Hub repo;
`adapters/` is gitignored) and the raw dev/train prompts.

Runtime → Change runtime type → **GPU (A100)**. Needs Colab secrets `HF_TOKEN` (gated WildJailbreak +
the private adapter repo) and `GH_TOKEN` (clone the private repo).

In [1]:
# 1. GPU check
import platform

import torch

print("python :", platform.python_version())
print("torch  :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU    :", props.name)
    print("VRAM   :", round(props.total_memory / 1e9, 1), "GB")
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU (A100).")

python : 3.12.13
torch  : 2.11.0+cu128 | CUDA available: True
GPU    : NVIDIA A100-SXM4-40GB
VRAM   : 42.4 GB


In [2]:
# 2. Secrets + clone/update the (private) repo
import os
import stat
import subprocess
import tempfile

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]  # transformers/datasets read this
os.environ["GIT_TOKEN"] = userdata.get("GH_TOKEN")            # token stays in the ENV, never in argv
REPO = "kambleakash0/safestack-study"
DEST = "/content/safestack-study"
REPO_URL = f"https://github.com/{REPO}.git"                   # tokenless remote (no PAT in .git/config)

# Auth via a GIT_ASKPASS helper that READS the token from the environment: the script itself holds no
# secret, and the token reaches git through the (owner-only) process env, not a command-line argument
# (argv is world-readable via /proc/<pid>/cmdline), and never touches .git/config or disk.
_askpass = tempfile.NamedTemporaryFile("w", suffix=".sh", delete=False)
_askpass.write('#!/bin/sh\ncase "$1" in *[Uu]sername*) echo x-access-token ;; *) echo "$GIT_TOKEN" ;; esac\n')
_askpass.close()
os.chmod(_askpass.name, stat.S_IRWXU)                        # 0700, owner-only
_git_env = {**os.environ, "GIT_ASKPASS": _askpass.name, "GIT_TERMINAL_PROMPT": "0"}

# Clone if missing, else force the checkout to the latest main. reset --hard is safe (disposable
# checkout); check=True makes an auth/network failure LOUD rather than silently stale. try/finally so
# the token and the askpass helper are ALWAYS cleaned up -- even if a git op raises, the GH_TOKEN
# never lingers in the kernel env and no helper file is left on disk.
try:
    if not os.path.isdir(DEST):
        subprocess.run(["git", "clone", "-q", REPO_URL, DEST], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "fetch", "-q", "origin", "main"], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", "-q", "origin/main"], check=True, env=_git_env)
finally:
    os.remove(_askpass.name)            # drop the askpass helper (even on failure)
    os.environ.pop("GIT_TOKEN", None)   # drop the token from the environment (even on failure)
%cd /content/safestack-study
!git log --oneline -1

/content/safestack-study
b72d098 (HEAD -> main, origin/main, origin/HEAD) fix(nb): remove Colab's torchao 0.10.0 so PEFT can load the LoRA adapter on a bf16 base (#83)


In [3]:
# 3. Install SafeStack + the [train] extra (LoRA/QLoRA: bitsandbytes + accelerate; peft via [hf])
!pip -q install -e ".[train]"
# Colab preinstalls torchao 0.10.0, which the newer PEFT rejects (needs > 0.16.0) and RAISES on when
# loading a LoRA adapter onto a non-4bit (bf16) base -- the SFT dev-eval and the C5-C8 evals. We use
# bitsandbytes, not torchao, so remove it: PEFT's is_torchao_available() then returns False and skips
# that dispatcher cleanly. (Training's 4-bit path already short-circuits to the bnb dispatcher first.)
!pip -q uninstall -y torchao
import peft
import transformers

print("transformers", transformers.__version__, "| peft", peft.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 67.4 MB/s eta 0:00:00
  Building editable for safestack (pyproject.toml) ... done
transformers 5.13.1 | peft 0.19.1


In [4]:
# 4. Mount Drive for resumable caches + adapter staging (a killed session resumes in minutes)
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/safestack"
CACHE = f"{BASE}/cache"
RUNS = f"{BASE}/runs"
ADAPTERS = f"{BASE}/adapters"          # a persistent copy of the adapter (also uploaded to HF-Hub)
REPORTS = "/content/safestack-study/reports"
ADAPTER_LOCAL = "adapters/sft_mistral_lora_v1"   # matches output_adapter in the train config
for d in (CACHE, RUNS, ADAPTERS, REPORTS):
    os.makedirs(d, exist_ok=True)
print("cache :", CACHE)
print("runs  :", RUNS)

Mounted at /content/drive
cache : /content/drive/MyDrive/safestack/cache
runs  : /content/drive/MyDrive/safestack/runs


In [5]:
# 5. Prepare all reference suites in dependency order -- the prep guards FAIL CLOSED on a partial set
#    (ADR-0015 follow-up 2): the 5 locked-test EVAL suites first, then train_sft (prepare-sft requires
#    every eval_* suite prepared, so it can dedup against all of them), then the 3 DEV suites (require
#    eval + train prepared). WildJailbreak is gated -> needs the HF token. A prepare failure STOPS here.
def _prep(cmd, label):
    print(f"--- {label} ---")
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-1500:], end="")
    if p.returncode != 0:
        print(p.stderr[-2000:])
        raise SystemExit(f"prepare failed: {label}")

EVAL_SUITES = [
    "harmful_advbench_v1",
    "harmful_harmbench_v1",
    "dualuse_harmbench_contextual_v1",   # eval_dual_use: ADR-0015's top-priority leakage-dedup target
    "overrefusal_xstest_v1",
    "helpfulness_alpaca_v1",
]
DEV_SUITES = [
    "dev_harmful_maliciousinstruct_v1",
    "dev_overrefusal_orbench_v1",
    "dev_helpfulness_alpaca_v1",
]
for name in EVAL_SUITES:
    _prep(["safestack", "data", "prepare", "-c", f"configs/datasets/{name}.yaml"], name)
_prep(["safestack", "data", "prepare-sft", "-c", "configs/datasets/sft_wildjailbreak_v1.yaml"],
      "train_sft (WildJailbreak, gated)")
for name in DEV_SUITES:
    _prep(["safestack", "data", "prepare", "-c", f"configs/datasets/{name}.yaml"], name)

--- harmful_advbench_v1 ---
prepared harmful_advbench_v1: 520 records -> sha256:a80ecfba71fadd12f194a658b924cbf6dd6b014f6b1d93057db4f422e2cfb4c3
--- harmful_harmbench_v1 ---
prepared harmful_harmbench_v1: 200 records -> sha256:1aabe6806d144c5d86ac03d64c77a6ba76f9446c0dc98833d300f24959f4b82f
--- dualuse_harmbench_contextual_v1 ---
prepared dualuse_harmbench_contextual_v1: 100 records -> sha256:52ced8ea6b4a8df5da1ad95793a00568ab517acb8a621df4e16f69dfede18ef7
--- overrefusal_xstest_v1 ---
prepared overrefusal_xstest_v1: 250 records -> sha256:24bd1fad943d9a368632b4b97d6d7f52aabda05a757c03c4dc8c87d3f6928fb6
--- helpfulness_alpaca_v1 ---
prepared helpfulness_alpaca_v1: 200 records -> sha256:31d0aa39d2f6d31294ee86a8b4829b24483434c6edcf8c01236ff30b93444d66
--- train_sft (WildJailbreak, gated) ---
prepared sft_wildjailbreak_v1: 10000 records -> sha256:34018e6c1356fd007fcaac2138ce6d9d288c566c1f8037e2b79fe2958c95a4fc
--- dev_harmful_maliciousinstruct_v1 ---
prepared dev_harmful_maliciousinstruct_

In [6]:
# 6. Drift guard (content-hash only). The manifests (data/manifests/) are committed with the
#    pinned-revision content hashes, and cell 5 just regenerated them. Compare only each manifest's
#    `hash` field against the committed pin: `created_at` is restamped to today's date on every prep,
#    so a whole-file `git diff` would false-positive on it. A real upstream drift (a pinned source
#    changed, or a parsing/tokenizer shift) changes the content hash -> STOP before train/eval.
import yaml

_manifests = EVAL_SUITES + ["sft_wildjailbreak_v1"] + DEV_SUITES
_drift = []
for _name in _manifests:
    _path = f"data/manifests/{_name}.yaml"
    _regen = yaml.safe_load(open(_path))["hash"]
    _committed = yaml.safe_load(
        subprocess.run(["git", "show", f"HEAD:{_path}"], capture_output=True, text=True).stdout
    )["hash"]
    if _regen != _committed:
        _drift.append(f"{_name}: committed {_committed} != regenerated {_regen}")
if _drift:
    print("\n".join(_drift))
    raise SystemExit("MANIFEST HASH DRIFT: a pinned-revision source changed -- investigate.")
print("no data drift: all", len(_manifests), "manifest content hashes match the committed pins")

no data drift: all 9 manifest content hashes match the committed pins


In [7]:
# 7. Leakage gate (ADR-0015 follow-up 2): train_sft must not overlap any eval OR dev suite
#    (exact + near-duplicate, Jaccard >= 0.7). Now that all 5 eval + 3 dev suites are prepared, this
#    checks against ALL of them (incl. eval_dual_use). Exits non-zero on overlap -> gate training on it.
gate = subprocess.run(
    ["safestack", "data", "overlap", "--train-split", "train_sft"],
    capture_output=True, text=True,
)
print(gate.stdout[-2000:])
if gate.returncode != 0:
    print(gate.stderr[-2000:])
    raise SystemExit("LEAKAGE GATE FAILED: train_sft overlaps an eval/dev suite -- do NOT train.")
print("leakage gate PASS")

train_sft: 10000 train records vs 8 eval suites -> 0 exact, 0 near-dup (>= 0.7)

leakage gate PASS


In [8]:
# 8. Train the LoRA/QLoRA SFT adapter (ADR-0015 dec.3): 1 epoch, r16/a32, 4-bit QLoRA, assistant-only
#    loss. The long GPU step (tens of minutes on an A100). Writes the adapter + committed loss curves.
import shutil

train = subprocess.run(
    ["safestack", "train", "sft", "-c", "configs/train/sft_mistral_lora_v1.yaml"],
    capture_output=True, text=True,
)
print(train.stdout[-3000:])
if train.returncode != 0:
    print(train.stderr[-4000:])
    raise SystemExit("training failed")
assert os.path.exists(f"{ADAPTER_LOCAL}/adapter_config.json"), "adapter not written"
# Stage a copy on Drive so a session death after training does not force a retrain.
shutil.copytree(ADAPTER_LOCAL, f"{ADAPTERS}/sft_mistral_lora_v1", dirs_exist_ok=True)
print("adapter at", ADAPTER_LOCAL, "(+ staged to Drive)")

5053'}
{'loss': '0.8323', 'grad_norm': '1.187', 'learning_rate': '9.836e-06', 'epoch': '0.5221'}
{'loss': '0.8573', 'grad_norm': '1.334', 'learning_rate': '9.292e-06', 'epoch': '0.5389'}
{'loss': '0.7999', 'grad_norm': '1.297', 'learning_rate': '8.749e-06', 'epoch': '0.5558'}
{'loss': '0.8439', 'grad_norm': '1.47', 'learning_rate': '8.21e-06', 'epoch': '0.5726'}
{'loss': '0.7665', 'grad_norm': '1.327', 'learning_rate': '7.676e-06', 'epoch': '0.5895'}
{'loss': '0.7989', 'grad_norm': '1.373', 'learning_rate': '7.149e-06', 'epoch': '0.6063'}
{'loss': '0.8017', 'grad_norm': '1.244', 'learning_rate': '6.631e-06', 'epoch': '0.6232'}
{'loss': '0.8133', 'grad_norm': '1.438', 'learning_rate': '6.123e-06', 'epoch': '0.64'}
{'loss': '0.8066', 'grad_norm': '1.328', 'learning_rate': '5.626e-06', 'epoch': '0.6568'}
{'loss': '0.7452', 'grad_norm': '1.418', 'learning_rate': '5.142e-06', 'epoch': '0.6737'}
{'loss': '0.7924', 'grad_norm': '1.184', 'learning_rate': '4.673e-06', 'epoch': '0.6905'}
{'loss'

In [9]:
# 9. Upload the adapter to a PRIVATE HF-Hub repo for an immutable id (ADR-0015 dec.7b). The weights
#    live here, never in the public git repo. The returned commit SHA is the adapter_revision to pin.
from huggingface_hub import HfApi, create_repo

# Must match `adapter:` in configs/models/sft_mistral_lora_v1.yaml (adjust to your HF namespace).
ADAPTER_REPO = "kambleakash0/safestack-sft-mistral-lora-v1"
create_repo(ADAPTER_REPO, private=True, repo_type="model", exist_ok=True, token=os.environ["HF_TOKEN"])
commit = HfApi().upload_folder(
    repo_id=ADAPTER_REPO,
    folder_path=ADAPTER_LOCAL,
    repo_type="model",
    commit_message="SFT LoRA adapter (sft_mistral_lora_v1)",
    token=os.environ["HF_TOKEN"],
)
ADAPTER_SHA = getattr(commit, "oid", None) or HfApi().model_info(
    ADAPTER_REPO, token=os.environ["HF_TOKEN"]
).sha
print("uploaded", ADAPTER_REPO, "@", ADAPTER_SHA, "(private)")
print("-> pin this SHA as adapter_revision in configs/models/sft_mistral_lora_v1.yaml for FU6 (C5-C8)")

# Pin the SHA into the LOCAL (disposable clone) card so THIS session's dev-eval loads the immutable
# commit, not the repo's mutable default branch. The COMMITTED card stays null -- pin it deliberately
# in the FU6 PR, never git-commit this in-session edit.
import pathlib
import re

_card = pathlib.Path("configs/models/sft_mistral_lora_v1.yaml")
_card.write_text(
    re.sub(r"(?m)^adapter_revision:.*$", f"adapter_revision: {ADAPTER_SHA}", _card.read_text())
)
print("pinned adapter_revision =", ADAPTER_SHA, "in the local card for this session's dev-eval")

uploaded kambleakash0/safestack-sft-mistral-lora-v1 @ 05266a9bd3fc1c75c515ea39ac5f7139abd77d31 (private)
-> pin this SHA as adapter_revision in configs/models/sft_mistral_lora_v1.yaml for FU6 (C5-C8)
pinned adapter_revision = 05266a9bd3fc1c75c515ea39ac5f7139abd77d31 in the local card for this session's dev-eval


In [10]:
# 10. Dev eval of the BASE (rule-9 reference) and the SFT policy (base+LoRA, NO guardrails = C5).
#     Generate + judge on the 3 DEV suites; the SFT card loads the private Hub adapter from step 9.
def _run_and_judge(cfg_name):
    r = subprocess.run(
        ["safestack", "eval", "run", "-c", f"configs/experiments/{cfg_name}.yaml",
         "--backend", "hf_local", "--cache-dir", CACHE, "--runs-dir", RUNS],
        capture_output=True, text=True,
    )
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise SystemExit(f"eval run failed: {cfg_name}")
    run = r.stdout.split("run:")[-1].strip().splitlines()[0]
    subprocess.run(
        ["safestack", "eval", "judge", "--run", run, "--kind", "all", "--cache-dir", CACHE],
        check=True,
    )
    return run

BASE_RUN = _run_and_judge("dev_selection_base")
SFT_RUN = _run_and_judge("dev_selection_sft")
print("BASE_RUN =", BASE_RUN, "\nSFT_RUN  =", SFT_RUN)

run: /content/drive/MyDrive/safestack/runs/7a394c6eb89b4b289773fe6505c0cbc0

run: /content/drive/MyDrive/safestack/runs/9f4286582e2f4a45b2b7e8fc5258b200

BASE_RUN = /content/drive/MyDrive/safestack/runs/7a394c6eb89b4b289773fe6505c0cbc0 
SFT_RUN  = /content/drive/MyDrive/safestack/runs/9f4286582e2f4a45b2b7e8fc5258b200


In [11]:
# 11. Rule-9 selection (ADR-0015 amendment 1): dev metrics -> tripwire + objective -> select.
#     Aggregate-only; writes the committed Selection + the 6 dev metrics artifacts (no raw text).
import json

from safestack.eval.artifacts import write_artifact
from safestack.eval.config import load_eval_config
from safestack.eval.metrics import suite_metrics
from safestack.train.select import checkpoint_dev_metrics, select_checkpoint, write_selection

SUITE_BY_ROLE = {
    "harmful": "dev_harmful_maliciousinstruct_v1",
    "overrefusal": "dev_overrefusal_orbench_v1",
    "helpfulness": "dev_helpfulness_alpaca_v1",
}

def _dev_metrics(run, cfg_name, label, step):
    cfg = load_eval_config(f"configs/experiments/{cfg_name}.yaml")
    arts = {}
    for role, suite in SUITE_BY_ROLE.items():
        art = suite_metrics(run, cfg, suite, cache_dir=CACHE)
        write_artifact(art, f"{REPORTS}/metrics/{cfg_name}__{suite}.json")  # aggregate-only provenance
        arts[role] = art
    return checkpoint_dev_metrics(
        label, step, harmful=arts["harmful"], overrefusal=arts["overrefusal"],
        helpfulness=arts["helpfulness"],
    )

base_dm = _dev_metrics(BASE_RUN, "dev_selection_base", "base", 0)
# The selected checkpoint records the immutable adapter identity (repo@sha) for provenance.
sft_dm = _dev_metrics(SFT_RUN, "dev_selection_sft", f"{ADAPTER_REPO}@{ADAPTER_SHA}", 1)

selection = select_checkpoint([sft_dm], base_dm)
out = write_selection(selection, f"{REPORTS}/selection/sft_mistral_lora_v1.json")
print(selection.rationale)
print("\nselection ->", out)
print(json.dumps(json.loads(selection.to_json()), indent=2))

Rule-9 checkpoint selection (ADR-0015 amendment 1) over 1 candidate(s).
Tripwire: dev-helpfulness answer-rate >= base 1.000 - 0.100 = 0.900; 1 survivor(s).
Objective ASR_dev + over_refusal_dev: selected 0.040 + 0.280 = 0.320 (CI [0.210, 0.450]); best-objective among 1 within-CI-noise contender(s), tiebroken by lower ASR then step.
Selected checkpoint 'kambleakash0/safestack-sft-mistral-lora-v1@05266a9bd3fc1c75c515ea39ac5f7139abd77d31' (step 1); base dev ASR 0.460, selected dev ASR 0.040.
Flags: none. Dev evidence degenerate: no.

selection -> /content/safestack-study/reports/selection/sft_mistral_lora_v1.json
{
  "asr": 0.04,
  "base_asr": 0.46,
  "base_asr_ci_high": 0.56,
  "base_asr_ci_low": 0.37,
  "base_helpfulness_answer_rate": 1.0,
  "dev_degenerate": false,
  "flags": [],
  "helpfulness_answer_rate": 0.96,
  "n_candidates": 1,
  "n_contenders": 1,
  "n_survivors": 1,
  "objective": 0.32,
  "objective_ci_high": 0.45,
  "objective_ci_low": 0.21,
  "over_refusal": 0.28,
  "rational

In [12]:
# 12. Loss curves (committed, aggregate-only): the tracked train + val loss from the run.
c = json.load(open("reports/train_curves/sft_mistral_lora_v1.json"))
print("n_train:", c["n_train"], "| n_val:", c["n_val"], "| precision:", c["hyperparameters"]["precision"])
print("final train loss:", c["final_train_loss"], "| final val loss:", c["final_val_loss"])
print("train points:", len(c["curves"]["train"]), "| val points:", len(c["curves"]["val"]))

n_train: 9500 | n_val: 500 | precision: bf16
final train loss: 0.7539703369140625 | final val loss: 0.8384018540382385
train points: 59 | val points: 1


## After the run

**Commit (aggregate-only)** from the repo, then push:
- `reports/selection/sft_mistral_lora_v1.json` — the rule-9 `Selection` (checkpoint, flags, rationale)
- `reports/metrics/dev_selection_*__*.json` — the 6 dev metrics artifacts
- `reports/train_curves/sft_mistral_lora_v1.json` — the loss curves
- this executed notebook (aggregate-only outputs; verify no raw prompts/generations)

Do **not** commit the in-session `adapter_revision` pin the notebook wrote into
`configs/models/sft_mistral_lora_v1.yaml` (step 9) -- that local edit only immutably pins the
*selection* run; `git restore` it.

**Pin the adapter (FU6):** in a separate FU6 PR, set `adapter_revision:` in the SFT card to the printed
commit SHA, so C5-C8 share one immutable policy identity (ADR-0015 dec.6/7b).

**Private, never committed:** the adapter weights (the private HF-Hub repo + `adapters/`) and the raw
dev/train prompts.

**Reading the gate:** if the `Selection` shows `dev_degenerate: true` (`no_tripwire_survivor` — the
adapter mode-collapsed on dev; or `no_asr_improvement` — dev showed no ASR reduction), go into the
locked-test C5 read (decision 5) expecting a DEGENERATE / NULL outcome, and interpret it there — never
on dev.